In [ ]:
!pip install transformers accelerate pillow -q

from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from PIL import Image
import torch, time, pandas as pd
from datasets import load_dataset

# Load model — 2B fits on free T4
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

PROMPT = "Extract all text from this table in row-column order. Return plain text only."

ds = load_dataset("bsmock/pubtables-1m", split="train[:300]")
results = []

for i, sample in enumerate(ds):
    img = sample['image']
    messages = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text",  "text": PROMPT}
    ]}]
    text_input = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=[text_input], images=[img], return_tensors="pt").to("cuda")

    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512)
    elapsed = time.perf_counter() - t0

    vlm_text = processor.decode(out[0], skip_special_tokens=True)
    results.append({'id': i, 'vlm_text': vlm_text, 'vlm_time': round(elapsed, 3)})
    if i % 25 == 0: print(f"Done {i}/300")

pd.DataFrame(results).to_csv("results/vlm_results.csv", index=False)